# Tokenizer Deep Dive

Tokenizer'ın A'dan Z'ye incelenmesi:

1. **Tokenization nedir?** — Neden gerekli, temel kavramlar
2. **Byte Pair Encoding (BPE)** — Algoritma, adım adım çalışma mantığı
3. **BPE Sıfırdan Implementasyon** — Merge'ler, vocab oluşturma
4. **Tokenizer Training** — Vocab size seçimi, training süreci
5. **Gerçek Tokenizer Karşılaştırma** — tiktoken (GPT-2) vs Qwen3.5 tokenizer
6. **Özel Tokenlar** — Special tokens, chat template, system/user/assistant
7. **Tokenizer'ı Genişletme** — Yeni token ekleme, embedding güncelleme
8. **Pratik Konular** — Token sayısı optimizasyonu, çok dilli tokenization, edge case'ler

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

---
## 1. Tokenization Nedir?

LLM'ler doğrudan metin işleyemez — sayılarla çalışır.
Tokenization: **metin → token ID'leri** dönüşümü.

```
"Hello world" → [15339, 1917] → Embedding → Model → Logits → [next_token_id] → "!"
```

**Üç yaklaşım:**
- **Karakter bazlı**: Her karakter bir token. Vocab küçük (~256) ama sequence çok uzun.
- **Kelime bazlı**: Her kelime bir token. Vocab çok büyük, bilinmeyen kelimeler sorunlu.
- **Subword (BPE)**: Orta yol — sık kelimeler tek token, nadir kelimeler parçalanır.

In [ ]:
text = "Tokenization is fundamental to LLMs."

# Karakter bazlı
char_tokens = list(text)
print(f"Karakter bazlı : {len(char_tokens)} token")
print(f"  {char_tokens[:15]}...")

# Kelime bazlı
word_tokens = text.split()
print(f"\nKelime bazlı   : {len(word_tokens)} token")
print(f"  {word_tokens}")

# Byte bazlı (UTF-8)
byte_tokens = list(text.encode("utf-8"))
print(f"\nByte bazlı     : {len(byte_tokens)} token")
print(f"  {byte_tokens[:15]}...")
print(f"  Her byte 0-255 arası → vocab size = 256")

In [ ]:
# UTF-8 encoding: farklı dillerde byte sayısı değişir
examples = [
    ("Hello", "İngilizce (ASCII)"),
    ("Merhaba", "Türkçe"),
    ("你好", "Çince"),
    ("🤖", "Emoji"),
]

for text, lang in examples:
    bytes_repr = list(text.encode("utf-8"))
    print(f"  '{text}' ({lang}): {len(text)} karakter, {len(bytes_repr)} byte → {bytes_repr}")

---
## 2. Byte Pair Encoding (BPE) — Algoritma

BPE, en sık görülen byte çiftlerini iteratif olarak birleştirir:

1. Metni byte'lara çevir (256 base token)
2. En sık geçen ardışık çifti bul
3. Bu çifti yeni bir token olarak birleştir (vocab'a ekle)
4. İstenen vocab size'a ulaşana kadar 2-3'ü tekrarla

Bu şekilde **merge kuralları** oluşur. Encoding sırasında bu kurallar sırayla uygulanır.

In [ ]:
# BPE adım adım gösterim
text = "aaabdaaabac"
tokens = list(text.encode("utf-8"))  # byte'lara çevir

print(f"Başlangıç: {tokens}")
print(f"           {''.join(chr(t) for t in tokens)}")
print()

def get_most_frequent_pair(token_list):
    pairs = Counter(zip(token_list[:-1], token_list[1:]))
    return pairs.most_common(1)[0] if pairs else None

def merge_pair(token_list, pair, new_id):
    result = []
    i = 0
    while i < len(token_list):
        if i < len(token_list) - 1 and (token_list[i], token_list[i+1]) == pair:
            result.append(new_id)
            i += 2
        else:
            result.append(token_list[i])
            i += 1
    return result

vocab = {i: bytes([i]) for i in range(256)}
merges = []
next_id = 256

for step in range(5):
    result = get_most_frequent_pair(tokens)
    if result is None:
        break
    pair, count = result
    
    vocab[next_id] = vocab[pair[0]] + vocab[pair[1]]
    merges.append(pair)
    tokens = merge_pair(tokens, pair, next_id)
    
    pair_repr = vocab[pair[0]].decode() + vocab[pair[1]].decode()
    print(f"Step {step+1}: merge ({pair[0]},{pair[1]}) = '{pair_repr}' → id={next_id}, count={count}")
    print(f"  Tokens: {tokens}")
    print(f"  Length: {len(tokens)}")
    print()
    
    next_id += 1

print(f"Merge kuralları: {merges}")
print(f"Vocab boyutu: 256 (base) + {len(merges)} (merges) = {256 + len(merges)}")

---
## 3. BPE Sıfırdan Implementasyon

Basit ama çalışan bir BPE tokenizer. İki aşama:
- **Training**: Metin üzerinde merge kurallarını öğren
- **Encoding/Decoding**: Merge kurallarını uygulayarak tokenize/detokenize et

In [ ]:
class SimpleBPE:
    def __init__(self):
        self.merges = {}       # (p1, p2) -> new_id
        self.vocab = {}        # id -> bytes
    
    def train(self, text, vocab_size):
        """BPE training: merge kurallarını öğren."""
        assert vocab_size >= 256
        tokens = list(text.encode("utf-8"))
        
        # Base vocab: 0-255 (single bytes)
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.merges = {}
        
        num_merges = vocab_size - 256
        for i in range(num_merges):
            # En sık çift
            pairs = Counter(zip(tokens[:-1], tokens[1:]))
            if not pairs:
                break
            top_pair = pairs.most_common(1)[0][0]
            
            new_id = 256 + i
            self.merges[top_pair] = new_id
            self.vocab[new_id] = self.vocab[top_pair[0]] + self.vocab[top_pair[1]]
            tokens = merge_pair(tokens, top_pair, new_id)
        
        return len(self.merges)
    
    def encode(self, text):
        """Metni token ID listesine çevir."""
        tokens = list(text.encode("utf-8"))
        # Merge kurallarını sırayla uygula
        for pair, new_id in self.merges.items():
            tokens = merge_pair(tokens, pair, new_id)
        return tokens
    
    def decode(self, token_ids):
        """Token ID listesini metne çevir."""
        byte_seq = b"".join(self.vocab[tid] for tid in token_ids)
        return byte_seq.decode("utf-8", errors="replace")

In [ ]:
# Training verisi indir
import requests

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
if not os.path.exists(file_path):
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(resp.text)

with open(file_path, "r", encoding="utf-8") as f:
    train_text = f.read()

print(f"Training text: {len(train_text):,} karakter")

# BPE training
bpe = SimpleBPE()
num_merges = bpe.train(train_text, vocab_size=500)
print(f"Merge sayısı: {num_merges}")
print(f"Vocab boyutu: {len(bpe.vocab)}")

In [ ]:
# Encode / Decode test
test_texts = [
    "Hello world",
    "The quick brown fox jumps over the lazy dog.",
    "Tokenization is important!",
]

for text in test_texts:
    ids = bpe.encode(text)
    decoded = bpe.decode(ids)
    tokens = [bpe.decode([tid]) for tid in ids]
    
    print(f"'{text}'")
    print(f"  IDs    : {ids}")
    print(f"  Tokens : {tokens}")
    print(f"  Decoded: '{decoded}'")
    print(f"  Sıkıştırma: {len(text.encode('utf-8'))} byte → {len(ids)} token ({len(text.encode('utf-8')) / len(ids):.1f}x)")
    print()

---
## 4. Tokenizer Training

Vocab size seçimi kritik:
- **Küçük vocab** (256-1K): Fazla token, uzun sequence, yavaş training
- **Büyük vocab** (100K+): Kısa sequence ama embedding tablosu çok büyük
- **Tipik değerler**: GPT-2: 50,257 | Llama: 32,000 | Qwen3.5: 248,320

In [ ]:
# Vocab size'ın token sayısına etkisi
test_text = "The quick brown fox jumps over the lazy dog. " * 10
vocab_sizes = [260, 300, 400, 500, 750, 1000]

results = []
for vs in vocab_sizes:
    t = SimpleBPE()
    t.train(train_text, vocab_size=vs)
    ids = t.encode(test_text)
    results.append((vs, len(ids)))
    print(f"  vocab_size={vs:>5d} → {len(ids):>4d} token (sıkıştırma: {len(test_text.encode('utf-8')) / len(ids):.1f}x)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([r[0] for r in results], [r[1] for r in results], marker="o", color="steelblue")
ax.set_xlabel("Vocab Size")
ax.set_ylabel("Token Sayısı")
ax.set_title("Vocab Size vs Token Sayısı (aynı metin)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Gerçek Tokenizer Karşılaştırma: tiktoken vs Qwen3.5

- **tiktoken** (GPT-2): OpenAI BPE, 50,257 vocab
- **Qwen3.5 tokenizer**: HuggingFace BPE, 248,320 vocab

In [ ]:
import tiktoken
from transformers import AutoTokenizer

# GPT-2 tokenizer (tiktoken)
gpt2_tok = tiktoken.get_encoding("gpt2")

# Qwen3.5 tokenizer
qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B-Base", trust_remote_code=True)

print(f"GPT-2 vocab size : {gpt2_tok.n_vocab:,}")
print(f"Qwen3.5 vocab size: {qwen_tok.vocab_size:,}")

In [ ]:
# Aynı metinleri her iki tokenizer ile tokenize et
test_texts = [
    ("Hello, how are you today?", "İngilizce"),
    ("Yapay zeka dünyayı değiştiriyor.", "Türkçe"),
    ("人工智能正在改变世界", "Çince"),
    ("def fibonacci(n):\n    if n < 2:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)", "Kod"),
    ("https://www.example.com/api/v2/users?page=1&limit=10", "URL"),
    ("2024-03-15T10:30:00Z", "Tarih"),
]

print(f"{'Metin':<30s} {'Dil':<10s} {'GPT-2':>6s} {'Qwen':>6s} {'Fark':>6s}")
print("-" * 65)

for text, lang in test_texts:
    gpt2_ids = gpt2_tok.encode(text)
    qwen_ids = qwen_tok.encode(text)
    diff = len(gpt2_ids) - len(qwen_ids)
    sign = "+" if diff > 0 else ""
    display = text[:28] + ".." if len(text) > 30 else text
    print(f"{display:<30s} {lang:<10s} {len(gpt2_ids):>6d} {len(qwen_ids):>6d} {sign}{diff:>5d}")

In [ ]:
# Detaylı token karşılaştırma
text = "Yapay zeka dünyayı değiştiriyor."

print(f"=== GPT-2 tokenization ===")
gpt2_ids = gpt2_tok.encode(text)
for tid in gpt2_ids:
    print(f"  {tid:>6d} → '{gpt2_tok.decode([tid])}'")

print(f"\n=== Qwen3.5 tokenization ===")
qwen_ids = qwen_tok.encode(text)
for tid in qwen_ids:
    print(f"  {tid:>6d} → '{qwen_tok.decode([tid])}'")

print(f"\nGPT-2 : {len(gpt2_ids)} token")
print(f"Qwen  : {len(qwen_ids)} token")

In [ ]:
# Token fertility: karakter başına token sayısı (düşük = daha verimli)
languages = {
    "English": "Artificial intelligence is transforming the world in unprecedented ways.",
    "Turkish": "Yapay zeka dünyayı benzeri görülmemiş şekillerde dönüştürüyor.",
    "Chinese": "人工智能正在以前所未有的方式改变世界。",
    "Arabic": "الذكاء الاصطناعي يغير العالم بطرق غير مسبوقة.",
    "Code": "for i in range(10): print(f'Hello {i}')",
}

print(f"{'Dil':<10s} {'GPT-2 fertility':>18s} {'Qwen fertility':>18s}")
print("-" * 50)
for lang, text in languages.items():
    g_ids = gpt2_tok.encode(text)
    q_ids = qwen_tok.encode(text)
    g_fertility = len(g_ids) / len(text)
    q_fertility = len(q_ids) / len(text)
    print(f"{lang:<10s} {g_fertility:>14.3f} tok/chr {q_fertility:>14.3f} tok/chr")

---
## 6. Özel Tokenlar (Special Tokens)

Special token'lar BPE merge sürecinden geçmez — doğrudan tek token olarak tanınır.

- `<|endoftext|>` — sequence sonu
- `<|im_start|>`, `<|im_end|>` — chat template
- `<think>`, `</think>` — reasoning mode (Qwen3)
- `<|vision_start|>` — multimodal tokenlar

In [ ]:
print("=== Qwen3.5 Special Tokens ===")
print(f"All special tokens: {qwen_tok.all_special_tokens}")
print(f"\nÖnemli tokenlar:")
for name in ['bos_token', 'eos_token', 'pad_token', 'unk_token']:
    token = getattr(qwen_tok, name)
    token_id = getattr(qwen_tok, f"{name}_id")
    print(f"  {name:<12s}: '{token}' (id={token_id})")

# Special token'ları encode etme
print(f"\n=== Special Token Encoding ===")
special_texts = [
    "Normal text",
    "<|im_start|>user\nHello<|im_end|>",
]
for text in special_texts:
    ids = qwen_tok.encode(text)
    tokens = [qwen_tok.decode([tid]) for tid in ids]
    print(f"  '{text[:50]}'")
    print(f"    → {ids}")
    print(f"    → {tokens}")

In [ ]:
# Chat template: model'e verilen gerçek input formatı
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is BPE?"},
]

if hasattr(qwen_tok, 'apply_chat_template'):
    chat_text = qwen_tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    print("=== Chat Template Output ===")
    print(repr(chat_text))
    
    print(f"\n=== Tokenized ===")
    chat_ids = qwen_tok.encode(chat_text)
    print(f"Token sayısı: {len(chat_ids)}")
    print(f"İlk 20 token:")
    for tid in chat_ids[:20]:
        print(f"  {tid:>6d} → '{qwen_tok.decode([tid])}'")
else:
    print("Chat template mevcut değil.")

---
## 7. Tokenizer'ı Genişletme

Yeni bir special token eklemek istediğimizde:
1. Tokenizer'a yeni token ekle
2. Model'in embedding katmanını genişlet
3. (Weight tying varsa) lm_head da güncellenir

Bu özellikle domain-specific tokenlar eklerken önemli (örn. `<|code_start|>`, `<|tool_call|>`).

In [ ]:
# tiktoken'a yeni special token ekleme
base_tok = tiktoken.get_encoding("gpt2")

# Yeni tokenlar
custom_tokens = {"<|tool_call|>": base_tok.n_vocab, "<|tool_result|>": base_tok.n_vocab + 1}

extended_tok = tiktoken.Encoding(
    name="gpt2_extended",
    pat_str=base_tok._pat_str,
    mergeable_ranks=base_tok._mergeable_ranks,
    special_tokens={**base_tok._special_tokens, **custom_tokens},
)

print(f"Original vocab: {base_tok.n_vocab:,}")
print(f"Extended vocab: {extended_tok.n_vocab:,}")

# Test
text = "Call function <|tool_call|> get_weather <|tool_result|> sunny"
ids = extended_tok.encode(text, allowed_special=set(custom_tokens.keys()))
print(f"\nEncoded: {ids}")
for tid in ids:
    print(f"  {tid:>6d} → '{extended_tok.decode([tid])}'")

In [ ]:
# Model embedding'ini genişletme (Ch05 yaklaşımı)
emb_dim = 768
old_vocab_size = 50257
new_vocab_size = 50259  # +2 yeni token

# Eski embedding simülasyonu
old_embedding = torch.nn.Embedding(old_vocab_size, emb_dim)

# Yeni embedding: büyütülmüş
new_embedding = torch.nn.Embedding(new_vocab_size, emb_dim)
new_embedding.weight.data[:old_vocab_size] = old_embedding.weight.data
# Yeni token'lar random init ile başlar

print(f"Eski embedding: {old_embedding}")
print(f"Yeni embedding: {new_embedding}")
print(f"\nYeni token embeddings (random init):")
print(f"  Token {old_vocab_size}: {new_embedding.weight.data[old_vocab_size, :5].tolist()}")
print(f"  Token {old_vocab_size+1}: {new_embedding.weight.data[old_vocab_size+1, :5].tolist()}")
print(f"\nBu yeni token'lar fine-tuning ile anlamlı embedding'ler öğrenir.")

---
## 8. Pratik Konular

### 8.1 Edge Case'ler

In [ ]:
# Edge case'ler: tokenizer'ın zor durumları
edge_cases = [
    "",                          # Boş string
    " ",                         # Sadece boşluk
    "\n\n\n",                    # Newline'lar
    "aaaaaaaaaaaa",              # Tekrarlayan karakter
    "a" * 100,                   # Çok uzun tekrar
    "🤖🤖🤖",                  # Emoji tekrar
    "123456789",                 # Sayılar
    "    def foo():    ",        # Kod boşlukları
    "<html><body></body></html>",# HTML
]

print(f"{'Metin':<35s} {'GPT-2':>6s} {'Qwen':>6s}")
print("-" * 50)
for text in edge_cases:
    g = gpt2_tok.encode(text)
    q = qwen_tok.encode(text)
    display = repr(text)[:33]
    print(f"{display:<35s} {len(g):>6d} {len(q):>6d}")

### 8.2 Token Sayısı ve Maliyet

API kullanırken token sayısı = maliyet. Daha verimli tokenizer = daha ucuz.

In [ ]:
# Büyük bir metin üzerinde token sayısı karşılaştırma
with open(file_path, "r", encoding="utf-8") as f:
    long_text = f.read()

g_ids = gpt2_tok.encode(long_text)
q_ids = qwen_tok.encode(long_text)

print(f"Metin uzunluğu  : {len(long_text):,} karakter")
print(f"GPT-2 token     : {len(g_ids):,}")
print(f"Qwen3.5 token   : {len(q_ids):,}")
print(f"Qwen tasarruf   : {(1 - len(q_ids)/len(g_ids))*100:.1f}%")
print(f"\nOrtalama token uzunluğu:")
print(f"  GPT-2 : {len(long_text) / len(g_ids):.1f} karakter/token")
print(f"  Qwen  : {len(long_text) / len(q_ids):.1f} karakter/token")

### 8.3 Tokenization ve Model Performansı

Tokenizer seçimi modelin performansını doğrudan etkiler:
- **Fertility oranı**: Aynı metin kaç tokena dönüyor? Düşük = daha iyi.
- **Çok dilli destek**: Vocab'da Türkçe/Çince token var mı? Yoksa her kelime parçalanır.
- **Kod desteği**: Boşluk, indent, özel semboller nasıl tokenize ediliyor?

In [ ]:
# Dil bazlı fertility görselleştirme
languages = {
    "English": "The quick brown fox jumps over the lazy dog.",
    "Turkish": "Hızlı kahverengi tilki tembel köpeğin üzerinden atlar.",
    "Chinese": "快速的棕色狐狸跳过了懒惰的狗。",
    "Arabic": "الثعلب البني السريع يقفز فوق الكلب الكسول.",
    "Russian": "Быстрая коричневая лиса перепрыгивает через ленивую собаку.",
    "Python": "def quick_brown_fox(): return 'jumps over lazy dog'",
    "JSON": '{"fox": "brown", "action": "jump", "over": "dog"}',
}

gpt2_ferts = []
qwen_ferts = []
langs = []

for lang, text in languages.items():
    g = len(gpt2_tok.encode(text)) / len(text)
    q = len(qwen_tok.encode(text)) / len(text)
    gpt2_ferts.append(g)
    qwen_ferts.append(q)
    langs.append(lang)

x = np.arange(len(langs))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, gpt2_ferts, width, label="GPT-2", color="steelblue")
ax.bar(x + width/2, qwen_ferts, width, label="Qwen3.5", color="coral")
ax.set_ylabel("Fertility (token/karakter, düşük = daha iyi)")
ax.set_xticks(x)
ax.set_xticklabels(langs, rotation=30)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title("Tokenizer Fertility Karşılaştırma")
plt.tight_layout()
plt.show()

---
## Özet

| Konu | Detay |
|---|---|
| **BPE Algoritması** | Byte çiftlerini iteratif birleştirme → merge kuralları |
| **Training** | Büyük corpus üzerinde merge kurallarını öğren |
| **Encoding** | Merge kurallarını sırayla uygula |
| **Decoding** | Token ID → byte sequence → metin |
| **Vocab Size** | Küçük = uzun sequence, Büyük = büyük embedding tablosu |
| **Special Tokens** | BPE'den geçmez, doğrudan tanınır |
| **Genişletme** | Yeni token + embedding resize + fine-tune |
| **Fertility** | Token/karakter oranı, dil ve tokenizer'a bağlı |

**Önemli çıkarımlar:**
- Tokenizer modelin görmediği en önemli bileşendir — kötü tokenizer = kötü model
- Vocab büyüklüğü trade-off: sequence kısalığı vs embedding boyutu
- Çok dilli modeller büyük vocab'a ihtiyaç duyar (Qwen: 248K vs GPT-2: 50K)
- Token eklerken embedding + lm_head birlikte güncellenmeli